# ViBioMIR trên Kaggle: từ corpus HF đến `submission.zip`

Notebook chạy **toàn bộ pipeline** trong một lần Run All, không cần clone repo:

1. `fetch`: tải `query.parquet`, `links_corpus.parquet` của BTC (revision cố định, kiểm tra SHA-256) và corpus đã crawl của nhóm (`corpus/*.parquet` trên Hugging Face). Commit của corpus được **ghim** cho run này; laptop chạy đúng lệnh này với cùng code.
2. `prepare` → `cards` (thẻ tài liệu: tiêu đề + đoạn mở đầu).
3. Một LLM (Qwen3-4B) dịch 1.200 truy vấn sang tiếng Trung, chạy theo batch. Chỉ dùng để **tìm**, không bao giờ được nộp.
4. **Truy xuất thác:** tìm trên thẻ của toàn corpus → giữ tài liệu ứng viên → chỉ chunk và lập chỉ mục đoạn cho các tài liệu đó.
5. **Một reranker** (bge-reranker-v2-m3), đọc đoạn tiếng Trung cùng bản dịch tiếng Trung của câu hỏi.
6. Chọn tài liệu/đoạn theo **ngưỡng tối ưu F2**, mở rộng đoạn nộp thành cửa sổ ≤ `WINDOW_TOKENS` (vẫn là đoạn trích nguyên văn), kiểm tra và đóng ZIP.

**Cài đặt notebook:** Accelerator **GPU T4 x2**, **Internet** bật, và Secret `HF_TOKEN` (Add-ons → Secrets) có quyền đọc dataset nhóm: dataset này **private**.

**Kết quả:** `/kaggle/working/submission.zip`. Pipeline giữ tối giản (một encoder, một reranker, một LLM dịch, không pilot) để dành thời gian cho nhiều dữ liệu hơn.

Code trong ô thứ 4 được sinh bởi `scripts/build_notebooks.py` từ repo. Sửa code trên laptop, build lại, upload lại; không sửa tay ô đó.

Thời gian chạy **chưa đo thật**; phụ thuộc số tài liệu trên HF. Để thử nhanh, đặt `LIMIT_ROWS = 20000` ở ô cấu hình.

In [ ]:
# ===== CẤU HÌNH =====
HF_CORPUS_REPO = 'AncientWings/AncientWings_R2AI_STAGE3_ds'   # corpus crawl của nhóm: corpus/*.parquet
HF_CORPUS_REVISION = None    # None: commit mới nhất (được ghim cho RUN); điền commit để chạy lại snapshot cũ

# Model cố định: encoder bge-m3, reranker bge-reranker-v2-m3, LLM dịch Qwen3-4B (fp16 trên T4).
HYDE = []                            # ['zh', 'vi'] để sinh thêm đoạn trả lời giả (chậm hơn)
DOCS_PER_QUERY = 200                 # tài liệu ứng viên mỗi truy vấn sau tầng thẻ
RERANK_K = 60                        # số đoạn reranker chấm mỗi truy vấn
WINDOW_TOKENS = 768                  # cửa sổ đoạn nộp; 0 = giữ nguyên chunk
LIMIT_ROWS = None                    # None: cả corpus; ví dụ 20000 để chạy thử
CALIBRATION = None                   # đường dẫn calibration.json (vibiomir.py evaluate calibrate) nếu có

from pathlib import Path
WORK = Path('/kaggle/working')
CODE_DIR = WORK / 'vibiomir_code'
RUN = WORK / 'run'   # chạy lại notebook trong cùng phiên: các khâu đã xong được dùng lại

In [ ]:
import importlib.metadata, os, subprocess, sys

def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

need = []
current = version('transformers')
major_minor = tuple(int(x) for x in current.split('.')[:2]) if current else (0, 0)
if not (4, 51) <= major_minor < (5, 0):
    need.append('transformers>=4.51,<5')   # Qwen3 cần >=4.51; code chưa kiểm thử với 5.x
if need:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('Đã nạp HF_TOKEN từ Kaggle Secrets.')
except Exception:
    print('CẢNH BÁO: không có Secret HF_TOKEN; dataset corpus của nhóm là private nên khâu fetch sẽ báo 401.')

import torch
GPUS = torch.cuda.device_count()
print('torch', torch.__version__, '| transformers', version('transformers'), '| GPU:',
      [torch.cuda.get_device_name(i) for i in range(GPUS)])
if GPUS == 0:
    raise RuntimeError('Không thấy GPU. Bật Accelerator GPU T4 x2 trong Settings.')

In [ ]:
# Sinh tự động bởi scripts/build_notebooks.py: code của repo, nhúng trong notebook.
# Không sửa tay ô này; sửa code trong repo rồi chạy lại script đó.
import base64, hashlib, io, shutil, zipfile
from pathlib import Path
CODE_SHA256 = '6986004c3f74a9babeb9b734f921defad28887380d738c6ad17c4fcfc04a43ae'
CODE_ZIP = (
    '...'
)
payload = base64.b64decode(CODE_ZIP)
if hashlib.sha256(payload).hexdigest() != CODE_SHA256:
    raise ValueError('Code nhúng bị hỏng; build lại notebook.')
shutil.rmtree(CODE_DIR, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(payload)).extractall(CODE_DIR)
print('Code:', CODE_SHA256[:12], sum(1 for _ in CODE_DIR.rglob('*.py')), 'file .py')


In [ ]:
# Chạy lại ô này sau khi bị ngắt: bước nào đã xong (kiểm tra SHA-256) được dùng lại.
command = [sys.executable, '-u', str(CODE_DIR / 'scripts' / 'vibiomir.py'), 'pipeline', '--profile', 'kaggle',
           '--gpus', str(min(GPUS, 2)), '--source', 'hf', '--hf-repo', HF_CORPUS_REPO,
           '--out', str(RUN), '--hyde', *HYDE, '--docs-per-query', str(DOCS_PER_QUERY), '--rerank-k', str(RERANK_K),
           '--window-tokens', str(WINDOW_TOKENS)]
if HF_CORPUS_REVISION:
    command += ['--hf-revision', HF_CORPUS_REVISION]
if LIMIT_ROWS:
    command += ['--limit', str(LIMIT_ROWS)]
if CALIBRATION:
    command += ['--calibration', str(CALIBRATION)]
env = {**os.environ, 'PYTHONIOENCODING': 'utf-8', 'TOKENIZERS_PARALLELISM': 'false'}
process = subprocess.Popen(command, cwd=CODE_DIR, env=env, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace')
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait():
    raise RuntimeError(f'Pipeline dừng với mã {process.returncode}; xem {RUN / "logs"}')

In [ ]:
import json, shutil
from IPython.display import FileLink, display

fetched = json.loads((RUN / 'hf' / 'fetch_manifest.json').read_text(encoding='utf-8'))
print(f"Dữ liệu: {fetched['corpus_repo']}@{fetched['corpus_revision'][:12]}, "
      f"{fetched['corpus_files']} file, {fetched['corpus_rows']:,} dòng; BTC {fetched['source_revision'][:12]}")

for name, target in (('submission', 'submission.zip'),):
    source = RUN / name / 'submission.zip'
    if source.exists():
        shutil.copyfile(source, WORK / target)
        manifest = json.loads((RUN / name / 'submission_manifest.json').read_text(encoding='utf-8'))
        print(f'\n== {target} ==')
        print(json.dumps({'submission': manifest['submission'],
                          'window_tokens': manifest['validation'].get('window_tokens'),
                          'chunk_tokens_median': manifest['validation'].get('chunk_tokens_median'),
                          'chunk_tokens_max': manifest['validation'].get('chunk_tokens_max'),
                          'warnings': manifest['validation']['warnings'][:5]},
                         ensure_ascii=False, indent=2))
        display(FileLink(str((WORK / target).relative_to(WORK))))
print('\nFile nộp: /kaggle/working/submission.zip')